In [15]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from softimpute_als import SoftImpute

def frob(U_old, D_old, V_old, U, D, V):
    old_norm_sq = (D_old ** 2).sum()
    new_norm_sq = (D ** 2).sum()

    utu = U_old.T @ U      # shape (J, J)
    vtv = V_old.T @ V      # shape (J, J)
    
    cross_mat = np.diag(D_old.flatten()) @ utu @ np.diag(D.flatten()) @ vtv
    cross_val = np.trace(cross_mat)  # scalar
    
    diff_sq = old_norm_sq + new_norm_sq - 2.0 * cross_val
    return diff_sq / max(old_norm_sq, 1e-9)

def soft_thresholding(sigma, lambd):
    return np.maximum(sigma - lambd, 0.0)

def svds_wrapper(A, k):
    """
    Computes the sparse SVD of A (using svds) and returns the singular vectors and
    singular values sorted in descending order.
    """
    if k >= min(A.shape):
        u, s, vt = np.linalg.svd(A, full_matrices=False)
        return u, s, vt
    else:
        print("=====use sparse")
        u, s, vt = svds(A, k=k)
        idx = np.argsort(s)[::-1]
        return u[:, idx], s[idx], vt[idx, :]


class SoftImputeSparse:
    def __init__(self, J=2, thresh=1e-5, lambd=0.0, maxit=100, random_state=None, verbose=False):
        """
        Sparse soft-impute that avoids forming a full dense imputed matrix.
        Assumes X is provided as a sparse CSR matrix storing only observed values.
        Missing entries are assumed to be absent.
        """
        self.J = J
        self.thresh = thresh
        self.lambd = lambd
        self.maxit = maxit
        self.rs = np.random.RandomState(random_state)
        self.verbose = verbose

        self.u = None
        self.d = None
        self.v = None

    def _compute_B(self, U, D, V, X, X_csc):
        """
        Compute B = U^T X_fill for each column j without forming X_fill.
        Here X_fill is defined by
            X_fill[i,j] = X[i,j]  if observed, and
                          U[i,:]·(D * V[j,:]) if missing.
        For each column j we use:
            B[:,j] = (U.T @ imputed_column) +
                     sum_{i in obs_j} U[i,:]*(X[i,j] - imputed_value[i])
        Notice that the full default sum can be computed as
            U.T @ imputed_column = Q · (D * V[j,:])
        where Q = U.T U.
        """
        n, J = U.shape
        m = V.shape[0]
        B = np.zeros((J, m))
        Q = U.T @ U  # J x J
        for j in range(m):
            # For column j, let dv = D * V[j,:]
            dv = D * V[j, :]  # shape (J,)
            default = Q.dot(dv)  # equals U.T @ (U @ (D*V[j,:]))
            # Get observed indices and values in column j (using CSC for fast column access)
            col = X_csc.getcol(j)
            obs_idx = col.indices
            obs_vals = col.data
            if obs_idx.size > 0:
                # For observed rows, compute imputed values: U[i,:]·(D*V[j,:])
                imputed_obs = U[obs_idx, :] @ (D * V[j, :])
                correction = U[obs_idx, :].T @ (obs_vals - imputed_obs)
            else:
                correction = 0.0
            B[:, j] = default + correction
        return B

    def _compute_A(self, U, D, V, X):
        """
        Compute A = X_fill V for each row i without forming X_fill.
        For each row i, X_fill[i,j] is defined as above.
        Then:
            A[i,:] = (sum_{j=1}^m V[j,:]*imputed_value[i,j])
                      + sum_{j in obs_i} V[j,:]*(X[i,j]-imputed_value[i,j])
        The default sum (over all j) can be written as:
            default = M · U[i,:],   with M = V.T @ (V * D)
        (Here V * D multiplies each column of V by the corresponding D element.)
        """
        n, _ = X.shape
        J = U.shape[1]
        A = np.zeros((n, J))
        # Precompute M = V.T @ (V * D)
        W = V * D[np.newaxis, :]  # shape (m, J)
        M = V.T @ W  # shape (J, J)
        for i in range(n):
            default = M.dot(U[i, :])
            row = X.getrow(i)  # efficient row slicing from CSR
            obs_idx = row.indices
            obs_vals = row.data
            if obs_idx.size > 0:
                # For each observed column j in row i, compute imputed value:
                # imputed = U[i,:]·(D*V[j,:])
                imputed_obs = np.array([U[i, :].dot(D * V[j, :]) for j in obs_idx])
                correction = V[obs_idx, :].T @ (obs_vals - imputed_obs)
            else:
                correction = 0.0
            A[i, :] = default + correction
        return A

    def fit(self, X):
        """
        Expects X as a sparse CSR matrix containing only observed entries.
        Missing entries are assumed to be absent.
        """
        # Ensure X is in CSR and also have a CSC copy for column access.
        if not hasattr(X, "tocsr"):
            raise ValueError("Input X must be a sparse matrix.")
        X = X.tocsr()
        X_csc = X.tocsc()
        n, m = X.shape

        # Initialize U with random values (n x J) and re-orthogonalize.
        U = self.rs.normal(0.0, 1.0, (n, self.J))
        U, _, _ = np.linalg.svd(U, full_matrices=False)
        # Initialize V (m x J) and D (vector length J)
        V = np.zeros((m, self.J))
        D = np.ones(self.J)

        ratio = 1.0
        iters = 0

        while ratio > self.thresh and iters < self.maxit:
            iters += 1
            U_old = U.copy()
            V_old = V.copy()
            D_old = D.copy()

            # --- Step 1: Fix V; update U and D using B = U.T X_fill ---
            B = self._compute_B(U, D, V, X, X_csc)  # B has shape (J, m)
            # Note: B.T is (m x J); m is moderate so we use full dense SVD.
            U_svd, s_svd, Vt_svd = np.linalg.svd(B.T, full_matrices=False)
            s_thr = soft_thresholding(s_svd, self.lambd)
            nonzero = s_thr > 1e-12
            r = np.sum(nonzero)
            if r == 0:
                V = np.zeros_like(V)
                D = np.zeros_like(D)
            else:
                s_thr = s_thr[:r]
                V_new = U_svd[:, :r]  # (m x r)
                D_new = s_thr         # (r,)
                Rmat = Vt_svd[:r, :]  # (r x J)
                # Update U (n x r)
                U = U @ Rmat.T
                # Pad with zeros if needed to maintain shape (n x J)
                if r < self.J:
                    pad_width = self.J - r
                    U = np.hstack([U, np.zeros((n, pad_width))])
                    V_new = np.hstack([V_new, np.zeros((m, pad_width))])
                    D_new = np.concatenate([D_new, np.zeros(pad_width)])
                V = V_new.copy()
                D = D_new.copy()

            # --- Step 2: Fix U; update V and D using A = X_fill V ---
            A = self._compute_A(U, D, V, X)  # A has shape (n, J)
            U_svd2, s_svd2, Vt_svd2 = np.linalg.svd(A, full_matrices=False)
            s_thr2 = soft_thresholding(s_svd2, self.lambd)
            nonzero2 = s_thr2 > 1e-12
            r2 = np.sum(nonzero2)
            if r2 == 0:
                U = np.zeros_like(U)
                D = np.zeros_like(D)
                V = np.zeros_like(V)
            else:
                s_thr2 = s_thr2[:r2]
                U_new = U_svd2[:, :r2]  # (n x r2)
                D_new = s_thr2         # (r2,)
                Rmat2 = Vt_svd2[:r2, :]  # (r2 x J)
                # Update V (m x r2)
                V = V @ Rmat2.T
                if r2 < self.J:
                    pad_width = self.J - r2
                    U_new = np.hstack([U_new, np.zeros((n, pad_width))])
                    D_new = np.concatenate([D_new, np.zeros(pad_width)])
                    V = np.hstack([V, np.zeros((m, pad_width))])
                U = U_new.copy()
                D = D_new.copy()

            ratio = frob(U_old, D_old, V_old, U, D, V)
            if self.verbose:
                print(f"Sparse iter: {iters:4d}, ratio = {ratio:.6f}")

        self.u = U[:, :self.J]
        self.d = D[:self.J]
        self.v = V[:, :self.J]
        return self

    def predict(self, X=None, copyto=False):
        """
        Returns the completed (imputed) matrix as a dense array:
             X_imp = U @ (diag(D) @ V^T).
        (X is ignored here since the completion is stored in the model.)
        """
        vd = self.v * self.d[np.newaxis, :]
        return self.u @ vd.T

# --- Example usage ---

def main():
    # Create a low–rank matrix and remove many entries.
    d1 = 1000
    d2 = 100
    r = 5
    p = 0.01  # probability of an observed entry
    # Generate a full dense low–rank matrix
    X_full = np.random.normal(2, 1, (d1, d2))
    U_true, D_true, Vt_true = np.linalg.svd(X_full, full_matrices=False)
    D_true[r:] = 0
    X_full = U_true.dot(np.diag(D_true)).dot(Vt_true)
    # Create an observation mask (only p fraction observed)
    mask = np.random.rand(d1, d2) < p
    # For the sparse input, we only store the observed entries.
    obs_i, obs_j = np.where(mask)
    obs_vals = X_full[obs_i, obs_j]

    M = X_full.copy()
    M[~mask] = np.nan
    print(M)

    clf = SoftImpute(J=2)
    fit = clf.fit(M)
    X_test = X_full.copy()
    X_imp = clf.predict(X_test)
    mse = np.mean((X_full[~mask] - X_imp[~mask])**2)
    print(mse)

    X_sparse = csr_matrix((obs_vals, (obs_i, obs_j)), shape=(d1, d2))

    print("Running sparse soft-impute...")
    clf_sparse = SoftImputeSparse(J=2, verbose=True, maxit=20)
    clf_sparse.fit(X_sparse)
    X_imp = clf_sparse.predict()

    # Evaluate performance on the previously missing entries.
    mse = np.mean((X_full[~mask] - X_imp[~mask])**2)
    print("Sparse mse on missing entries:", mse)
    XTX = X_full.T @ X_full
    XTX_imp = X_imp.T @ X_imp
    print("Relative norm difference of X^TX:", np.linalg.norm(XTX - XTX_imp) / np.linalg.norm(XTX))

if __name__ == '__main__':
    main()


[[nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]]
2.5000785716747242
Running sparse soft-impute...
Sparse iter:    1, ratio = 63.826453
Sparse iter:    2, ratio = 2.944750
Sparse iter:    3, ratio = 0.344214
Sparse iter:    4, ratio = 0.117923
Sparse iter:    5, ratio = 0.058612
Sparse iter:    6, ratio = 0.034682
Sparse iter:    7, ratio = 0.022518
Sparse iter:    8, ratio = 0.012446
Sparse iter:    9, ratio = 0.011148
Sparse iter:   10, ratio = 0.008304
Sparse iter:   11, ratio = 0.006371
Sparse iter:   12, ratio = 0.005012
Sparse iter:   13, ratio = 0.004027
Sparse iter:   14, ratio = 0.003295
Sparse iter:   15, ratio = 0.002738
Sparse iter:   16, ratio = 0.002305
Sparse iter:   17, ratio = 0.001962
Sparse iter:   18, ratio = 0.001686
Sparse iter:   19, ratio = 0.001461
Sparse iter:   20, ratio = 0.001275
Sparse mse on missing entries: 4.20996

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

